# 03 Applicant lists: snapshot and profile (T2.5a)

**Task:** T2.5a. Profiles the public applicant lists downloaded by
`harborfund.ingest.applicants` (MARAD PIDP, DOT TIGER–BUILD, DOT INFRA) and
`harborfund.ingest.openfema` (OpenFEMA HMA Subapplications v2, which holds BRIC and PDM).

**Reads:** `data/raw/applicants/<source>/<date>/`, `data/raw/openfema/<date>/`,
`data/manifest.csv`, `data/reference/coastal_counties.csv`,
`data/reference/program_inventory.csv`, and T1.1's national 97.047 USAspending pull
(`data/raw/usaspending/<date>/aln_97.047/`) for the award-ID cross-check.

**Writes:** nothing except through `harborfund.ingest.openfema.process`, which builds
`data/processed/bric_pacific_subapplicants.csv` for T1.3 (not committed).

Outputs are kept: they are part of the deliverable (the profile and the checks).

In [1]:
import json

import pandas as pd

from harborfund import config
from harborfund.ingest import applicants, openfema

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
PACIFIC = ["CA", "OR", "WA"]

## 1. Downloads logged in the manifest

T2.5a rows only. `changed_since_t04` compares with the checksums T0.4 recorded on 2026-11-17.

In [2]:
man = pd.read_csv(config.MANIFEST_PATH, dtype=str)
notes = man["notes"].map(json.loads)
t25 = man[notes.map(lambda n: n.get("task") == "T2.5a")].copy()
n = t25["notes"].map(json.loads)
t25["file"] = n.map(lambda d: d.get("file", "").rsplit("/", 1)[-1])
t25["bytes"] = n.map(lambda d: d.get("bytes"))
t25["rows"] = n.map(lambda d: d.get("rows"))
t25["pages"] = n.map(lambda d: d.get("pages"))
t25["changed_since_t04"] = n.map(lambda d: d.get("changed_since_t04"))
t25["page_last_updated"] = n.map(lambda d: d.get("page_last_updated"))
t25["sha256_12"] = t25["sha256"].str[:12]
t25[["source_id", "file", "retrieved_utc", "bytes", "rows", "pages", "sha256_12", "changed_since_t04", "page_last_updated"]]

,source_id,file,retrieved_utc,bytes,rows,pages,sha256_12,changed_since_t04,page_last_updated
17,openfema,HmaSubapplications_v2.csv,2026-11-18T16:18:56Z,28470485,63640,None,3a2a89be7b4b,None,None


## 2. OpenFEMA HMA Subapplications v2

National, every program and status. One row per subapplication (FEMA's unit), including
Not Selected / Not Approved. Subapplications that states screened out and never forwarded
to FEMA are absent (T0.4; asked for in FEMA FOIA Part B).

In [3]:
snap = openfema.read_snapshot()
cat = json.loads((config.ROOT / snap.attrs["snapshot_dir"] / "HmaSubapplications_v2_catalog.json").read_text())
print(snap.attrs["snapshot_dir"], f"{len(snap):,} rows;", "catalog recordCount", cat["recordCount"],
      "| lastRefresh", cat["lastRefresh"], "| lastDataSetRefresh", cat["lastDataSetRefresh"])
snap["program_id"] = openfema.hma_program_id(snap)
snap = snap.join(openfema.id_code(snap["subapplicationIdentifier"]))
snap.groupby("program", dropna=False).agg(rows=("program", "size"), fy_min=("fiscalYear", "min"), fy_max=("fiscalYear", "max")).sort_values("rows", ascending=False)

data/raw/openfema/2026-11-18 63,640 rows; catalog recordCount 63640 | lastRefresh 2026-07-27T22:39:30.327Z | lastDataSetRefresh 2026-11-18T12:14:36.641Z


,rows,fy_min,fy_max
program,,,
Hazard Mitigation Grant Program,42169,1989,2025
Pre-Disaster Mitigation,7922,2000,2026
Flood Mitigation Assistance,5980,1996,2024
Building Resilient Infrastructure and Communities,4871,2020,2025
Hazard Mitigation Grant Program Post Fire,989,2017,2025
Legislative Pre-Disaster Mitigation,414,2008,2010
Flood Mitigation Assistance Swift Current,407,2022,2024
Pre-Disaster Mitigation Congressionally Directed Spending,319,2022,2024
Severe Repetitive Loss,306,2008,2012


### 2a. Check (a): `program` strings for BRIC and PDM, and (b): earmarked PDM and legacy LPDM

ID family = the first letters of `subapplicationIdentifier` (FEMA GO IDs look like
`EMF-2020-BR-143-0016`; eGrants IDs like `PDMC-PJ-09-CA-2019-001` or `LPDM-PJ-10-OR-2024-001`;
NEMIS disaster IDs like `DR-1234-...`).

In [4]:
fam = snap.assign(id_kind=snap["id_code"].where(snap["id_code"] != "", snap["id_family"]))
bp = fam[snap["program"].isin(openfema.BRIC_PDM_PROGRAMS)]
bp.groupby(["program", "id_kind", "program_id"]).agg(rows=("program", "size"), fy_min=("fiscalYear", "min"), fy_max=("fiscalYear", "max"))

rows  fy_min  fy_max
program                                            id_kind program_id                           
Building Resilient Infrastructure and Communities  BR      bric             4871    2020    2025
Legislative Pre-Disaster Mitigation                LPDM    lpdm_legacy       414    2008    2010
Pre-Disaster Mitigation                            DR      pdm_competitive   515    2000    2004
                                                   PD      pdm_cpf           142    2026    2026
                                                   PDMC    pdm_competitive  7265    2003    2019
Pre-Disaster Mitigation Congressionally Directe... LPDM    pdm_cpf           319    2022    2024

In [5]:
# FY2026 "Pre-Disaster Mitigation" records: FEMA GO IDs with code PD, submitted 20-22 July 2026
pd26 = snap[(snap["program"] == openfema.PDM) & (snap["fiscalYear"] == 2026)]
print(len(pd26), "records;", pd26["id_award"].nunique(), "award IDs;", pd26["subapplicantStateAbbreviation"].nunique(), "states")
print("dateSubmittedToFema:", pd26["dateSubmittedToFema"].min(), "to", pd26["dateSubmittedToFema"].max())
print(pd26["status"].value_counts().to_string())
print("federal share $M:", round(pd26["federalShareAmount"].sum() / 1e6, 1))

142 records; 41 award IDs; 40 states
dateSubmittedToFema: 2026-07-06T00:00:00.000Z to 2026-07-22T00:00:00.000Z
status
Subaward Accepted    108
Submitted to FEMA     27
Subaward Offered       7
federal share $M: 167.8


These match FEMA's FY2026 PDM Congressionally Directed Spending NOFO (DHS-26-MT-143-00-01,
ALN **97.143**, applications due 22 July 2026, $189.7M for 125 named projects). So FY2026
earmarks carry the plain program string "Pre-Disaster Mitigation"; `program_id` separates
them by ID code PD and year ≥ 2022. The FY2022–24 earmarks use the program string
"Pre-Disaster Mitigation Congressionally Directed Spending" and eGrants `LPDM-` IDs, the
same prefix as the FY2008–10 "Legislative Pre-Disaster Mitigation" records.

In [6]:
# Status by program_id (outcome field) for the BRIC/PDM family
bp = snap[snap["program_id"] != ""]
pd.crosstab(bp["status"].fillna("(blank)"), bp["program_id"], margins=True)

program_id,bric,lpdm_legacy,pdm_competitive,pdm_cpf,All
status,,,,,
Approved,0,0,5,0,5
Closed,110,278,4476,8,4872
Closeout Initiated,140,0,0,0,140
Ineligible,1339,0,0,0,1339
Not Approved / Denied,0,131,2447,16,2594
Not Selected,597,0,686,0,1283
Obligated,0,1,153,290,444
Pending,0,4,0,4,8
Revision Requested,0,0,3,1,4


In [7]:
# BRIC selectionStatus (FEMA's national selection outcome) by fiscal year
bric = snap[snap["program_id"] == "bric"]
pd.crosstab(bric["selectionStatus"].fillna("(blank)"), bric["fiscalYear"], margins=True)

fiscalYear,2020,2021,2022,2023,2024,2025,All
selectionStatus,,,,,,,
(blank),0,0,0,0,97,954,1051
Did Not Meet HMA Requirements,156,208,154,194,0,0,712
Identified for Further Review,408,371,518,720,0,0,2017
Not Selected,423,210,129,321,0,0,1083
Withdrawn,6,0,2,0,0,0,8
All,993,789,803,1235,97,954,4871


### 2b. Rows by program × FY × state (BRIC/PDM family), and the CA/OR/WA subset

In [8]:
by_fy = pd.crosstab(bp["fiscalYear"], bp["program_id"], margins=True)
by_fy

program_id,bric,lpdm_legacy,pdm_competitive,pdm_cpf,All
fiscalYear,,,,,
2000,0,0,176,0,176
2002,0,0,148,0,148
2003,0,0,581,0,581
2004,0,0,91,0,91
2005,0,0,821,0,821
2006,0,0,230,0,230
2007,0,0,471,0,471
2008,0,207,485,0,692
2009,0,101,485,0,586


In [9]:
pac = bp[bp["subapplicantStateAbbreviation"].isin(PACIFIC)]
pd.crosstab([pac["program_id"], pac["fiscalYear"]], pac["subapplicantStateAbbreviation"], margins=True)

subapplicantStateAbbreviation   CA   OR   WA   All
program_id      fiscalYear                        
bric            2020            42    6   36    84
                2021            25    7   27    59
                2022            37   22   29    88
                2023            29   32   29    90
                2024             4    0   34    38
                2025            17   29   26    72
lpdm_legacy     2008            19    0   11    30
                2009            10    0    2    12
                2010            11    0    1    12
pdm_competitive 2003            38   10   13    61
                2005           140   18   28   186
                2006            15    7   10    32
                2007            53    9    6    68
                2008            72    7   13    92
                2009            51    5   14    70
                2010            52    6    8    66
                2011            18    2   13    33
                2012            43    5    8    56
                2013             6    5    7    18
                2014            18    4   15    37
                2015            15    6   11    32
                2016            26    8    8    42
                2017            32    4   31    67
                2018            34    9   32    75
                2019            41    4   26    71
pdm_cpf         2022             3    2    4     9
                2023            11    0    5    16
                2024            13    2    4    19
                2026            16    0    7    23
All                            891  209  458  1558

In [10]:
# Full program x FY x state table for every state (long form; saved nowhere, shown summarised)
long = bp.groupby(["program_id", "fiscalYear", "subapplicantStateAbbreviation"]).size().rename("rows").reset_index()
print(f"{len(long):,} program x FY x state cells; {long['subapplicantStateAbbreviation'].nunique()} states/territories")
long.pivot_table(index="subapplicantStateAbbreviation", columns="program_id", values="rows", aggfunc="sum", fill_value=0).loc[PACIFIC]

1,274 program x FY x state cells; 56 states/territories


program_id,bric,lpdm_legacy,pdm_competitive,pdm_cpf
subapplicantStateAbbreviation,,,,
CA,154,40,654,43
OR,96,0,109,4
WA,181,14,243,20


### 2c. Fields present (share non-blank, %), by program_id

Identifiers: UEI, name, city, ZIP, benefiting counties, type. Amounts:
`totalSubapplicationAmount` and `federalShareAmount` are requested/approved;
`totalObligatedAmount` / `totalPaymentAmount` are filled only once a subaward is accepted.

In [11]:
FIELDS = ["subapplicantUei", "subapplicantName", "subapplicantCity", "subapplicantZipCode", "benefitingCounties",
          "subapplicantType", "subapplicationTitle", "primaryActivity", "totalSubapplicationAmount", "federalShareAmount",
          "totalObligatedAmount", "totalPaymentAmount", "benefitCostRatio", "status", "selectionStatus", "dateSubmittedToFema"]
win = bp[bp["fiscalYear"] >= config.STUDY_FY_START]
(win.groupby("program_id")[FIELDS].agg(lambda s: s.notna().mean() * 100).round(0).T)

program_id,bric,lpdm_legacy,pdm_competitive,pdm_cpf
subapplicantUei,97.0,0.0,0.0,30.0
subapplicantName,100.0,100.0,100.0,100.0
subapplicantCity,100.0,100.0,100.0,100.0
subapplicantZipCode,100.0,100.0,100.0,100.0
benefitingCounties,100.0,76.0,79.0,88.0
subapplicantType,100.0,100.0,100.0,100.0
subapplicationTitle,80.0,0.0,0.0,0.0
primaryActivity,100.0,100.0,100.0,100.0
totalSubapplicationAmount,100.0,100.0,100.0,100.0
federalShareAmount,100.0,100.0,100.0,100.0


### 2d. Known quirks

- `benefitingCounties` gives county names without states, and some Pacific records list
  counties from other states (e.g. "CLATSOP; WARREN", "SAN MATEO; PLEASANTS; GASTON").
  The coastal flag therefore matches names only within the subapplicant's own state.
- `subapplicantCity` sometimes holds a street address (e.g. "3322 236th Street NE").
- PDM FY2010–19 and LPDM records have no title, UEI or `selectionStatus`.
- FEMA: the financial fields "are not derived from FEMA's official financial systems".
- The catalog `lastRefresh` (2026-07-27) predates `lastDataSetRefresh` (2026-11-18); the
  record count has not changed since T0.4 (63,640).

In [12]:
# Stray county names: Pacific BRIC/PDM records whose counties are not coastal in their own state
ext = openfema.pacific_extract(snap)
other = ext["counties_other"].dropna().str.split(";").explode().str.strip()
print("county names on Pacific records that are not coastal counties of the record's state (top 12):")
print(other.str.upper().str.replace(" COUNTY", "").value_counts().head(12).to_string())
addr_city = ext["subapplicantCity"].fillna("").str.match(r"^\d+\s")
print("\nPacific records whose city field starts with a street number:", int(addr_city.sum()))

county names on Pacific records that are not coastal counties of the record's state (top 12):
counties_other
                  929
SAN BERNARDINO     76
WASHINGTON         59
RIVERSIDE          57
CLACKAMAS          51
MULTNOMAH          45
ST. LOUIS          35
BENTON             34
FRANKLIN           32
DOUGLAS            29
GRANT              27
CLARK              26

Pacific records whose city field starts with a street number: 1


## 3. Check (c): T1.1's 97.047 award-ID rule against OpenFEMA

T1.1 classified USAspending 97.047 awards by FEMA award ID: `BR` = bric, `PC` =
pdm_competitive, `PD` with round ≥ FY2022 = pdm_cpf, earlier `PD` = pdm_other. FEMA GO award
IDs can be joined to OpenFEMA subapplication IDs (exactly, or on the first four parts).
eGrants-era awards cannot.

In [13]:
usa_dir = config.RAW_DIR / "usaspending"
f97 = sorted((usa_dir).glob("*/aln_97.047/*/Assistance_PrimeAwardSummaries_*.csv"))
print(f97[-1].relative_to(config.ROOT))
prime = pd.read_csv(f97[-1], dtype=str, low_memory=False)
cc = openfema.crosscheck_award_ids(prime, snap)
pd.crosstab([cc["fema_program_code"], cc["t11_program_id"]], cc["of_join"], margins=True)

data/raw/usaspending/2026-11-18/aln_97.047/PrimeAwardTransactionsAndSubawards_2026-11-18_H14M23S12859327/Assistance_PrimeAwardSummaries_2026-11-18_H14M23S16_1.csv


of_join                            award_prefix  exact  none   All
fema_program_code t11_program_id                                  
BR                bric                      189    646     0   835
CA                other_97047                 0      0     2     2
FM                other_97047                 2      0     0     2
GR                other_97047                 0      0     1     1
PC                pdm_competitive             0      0   358   358
PD                pdm_cpf                     0      0    64    64
                  pdm_other                   0      0    13    13
All                                         191    646   438  1275

In [14]:
joined = cc[cc["of_join"] != "none"]
print("Joined awards, T1.1 class vs OpenFEMA program_id ('' = an HMA program outside BRIC/PDM):")
print(joined.groupby(["t11_program_id", "of_program_id"]).size().to_string())

Joined awards, T1.1 class vs OpenFEMA program_id ('' = an HMA program outside BRIC/PDM):
t11_program_id  of_program_id
bric            bric             835
other_97047                        2


In [15]:
# PD awards from FY2022 on vs OpenFEMA earmark records, by year (state counts and dollars)
amt = lambda s: round(pd.to_numeric(s, errors="coerce").sum() / 1e6, 1)
usa_pd = cc[cc["t11_program_id"] == "pdm_cpf"].groupby("fema_round_fy").agg(
    usa_awards=("award_id_fain", "size"), usa_states=("recipient_state_code", "nunique"),
    usa_obligated_m=("total_obligated_amount", amt))
of_cpf = snap[snap["program_id"] == "pdm_cpf"].groupby("fiscalYear").agg(
    of_records=("program_id", "size"), of_states=("subapplicantStateAbbreviation", "nunique"),
    of_fed_share_m=("federalShareAmount", amt))
usa_pd.join(of_cpf, how="outer")

,usa_awards,usa_states,usa_obligated_m,of_records,of_states,of_fed_share_m
fema_round_fy,,,,,,
2022,29.0,27.0,144.1,70,27,134.7
2023,34.0,32.0,217.6,127,32,222.9
2024,1.0,1.0,5.7,122,36,180.4
2026,NaN,NaN,NaN,142,40,167.8


In [16]:
# PC awards: FAIN year vs description (eGrants-era PDM; cannot be joined to OpenFEMA)
pc = cc[cc["fema_program_code"] == "PC"]
desc = pc["prime_award_base_transaction_description"].fillna("").str.upper()
pd.crosstab(pc["fema_round_fy"], desc.str.contains("BRIC").map({True: "described as BRIC", False: "PDM / other"}), margins=True)

prime_award_base_transaction_description,PDM / other,described as BRIC,All
fema_round_fy,,,
2008,1,0,1
2010,3,0,3
2011,5,0,5
2012,7,0,7
2013,1,0,1
2014,16,0,16
2015,22,0,22
2016,47,0,47
2017,52,0,52


In [17]:
pc[desc.str.contains("BRIC")][["award_id_fain", "recipient_name", "recipient_state_code", "total_obligated_amount"]]

,award_id_fain,recipient_name,recipient_state_code,total_obligated_amount
462,EMD-2018-PC-0008,TURTLE MOUNTAIN BAND OF CHIPPEWA INDIANS,ND,-1.00
594,EMD-2022-PC-0001,NORTHERN CHEYENNE TRIBE,MT,71471.26
718,EMF-2022-PC-0001,BLUE LAKE RANCHERIA,CA,562356.00
719,EMF-2022-PC-0002,KARUK TRIBE HOUSING AUTHORITY,CA,810885.24
734,EMF-2024-PC-0001,PAUMA BAND OF MISSION INDIANS,CA,126000.00
1103,EMS-2021-PC-0013,TWIN HILLS VILLAGE COUNCIL,AK,179854.00
1104,EMS-2021-PC-0014,NEW STUYAHOK VILLAGE,AK,0.00
1112,EMS-2022-PC-0001,PUYALLUP TRIBE OF INDIANS,WA,167143.99
1113,EMS-2022-PC-0002,MAKAH INDIAN TRIBE OF THE MAKAH INDIAN RESERVATION,WA,108116.00


## 4. PDM FY2010–19 linkability

Titles and UEIs are blank. What else can identify the applicant?

In [18]:
pdm = snap[(snap["program_id"] == "pdm_competitive") & snap["fiscalYear"].between(2010, 2019)]
pdm_pac = pdm[pdm["subapplicantStateAbbreviation"].isin(PACIFIC)]
ID_FIELDS = ["subapplicantName", "subapplicantCity", "subapplicantZipCode", "benefitingCounties", "subapplicantType",
             "subapplicantUei", "subapplicationTitle", "primaryActivity"]
fill = pd.DataFrame({"national": pdm[ID_FIELDS].notna().mean() * 100, "CA/OR/WA": pdm_pac[ID_FIELDS].notna().mean() * 100}).round(0)
print(len(pdm), "national records;", len(pdm_pac), "in CA/OR/WA")
fill

4292 national records; 497 in CA/OR/WA


,national,CA/OR/WA
subapplicantName,100.0,100.0
subapplicantCity,100.0,100.0
subapplicantZipCode,100.0,100.0
benefitingCounties,79.0,77.0
subapplicantType,100.0,100.0
subapplicantUei,0.0,0.0
subapplicationTitle,0.0,0.0
primaryActivity,100.0,100.0


In [19]:
print(pdm_pac["subapplicantType"].value_counts().to_string())
print()
print(pdm_pac["status"].value_counts().to_string())
print("\ndistinct subapplicant names (CA/OR/WA):", pdm_pac["subapplicantName"].str.upper().str.strip().nunique())
pdm_pac[["fiscalYear", "subapplicantName", "subapplicantCity", "subapplicantZipCode", "benefitingCounties", "primaryActivity", "status"]].sample(8, random_state=3)

subapplicantType
Local Government                 252
Indian Tribal Government         136
State Government                  54
Special Governmental District     48
Private Non-Profit                 4
other                              3

status
Closed                   281
Not Approved / Denied    137
Not Selected              60
Obligated                 19

distinct subapplicant names (CA/OR/WA): 310


,fiscalYear,subapplicantName,subapplicantCity,subapplicantZipCode,benefitingCounties,primaryActivity,status
49316,2013,City of Yucaipa,Yucaipa,92399,SAN BERNARDINO,Retrofit,Closed
54669,2011,Spokane County,Spokane,99217,SPOKANE,New plan,Closed
5645,2010,Tahoe Truckee Unified School District,Truckee,96161,None,Other,Not Selected
45344,2012,City of Paso Robles,Paso Robles,93446,SAN LUIS OBISPO,New plan,Closed
27635,2014,Ventura County Sheriff's Office of Emergency Services,Ventura,93009,VENTURA; MILLARD; ANDREW,New plan,Closed
60587,2017,Stillaguamish Tribe of Indians,Arlington,98223,SNOHOMISH,Management costs,Closed
53117,2016,California Governors Office of Emergency Services,Mather,95655,None,Management costs,Closed
3151,2018,City of Seattle,Seattle,98104,KING,Retrofit,Closed


In [20]:
# Harbor-related wording is only available through primaryActivity / subactivity (no titles)
acts = pdm_pac["primaryActivity"].value_counts()
print(acts.head(15).to_string())

primaryActivity
New plan                                 127
Retrofit                                  72
Management costs                          62
Plan update                               51
Utility and infrastructure protection     43
Flood control                             39
Other                                     37
Generator                                 29
Wildfire management                       15
Acquisition                                8
Stabilization and restoration              6
Elevation                                  4
Saferoom/shelter                           3
Relocation                                 1


## 5. Extract for T1.3: `data/processed/bric_pacific_subapplicants.csv`

CA/OR/WA BRIC and PDM subapplications (competitive and earmarked, all years, all statuses),
with UEI, name, county, title, status, amounts, `any_coastal_county` (a listed benefiting
county is in `coastal_counties.csv`, matched within the subapplicant's state) and
`zip_in_coastal_county` (subapplicant ZIP → county via the Census ZCTA file from T1.1).

In [21]:
ext = openfema.process(None)

snapshot data/raw/openfema/2026-11-18: 63,640 rows
wrote data/processed/bric_pacific_subapplicants.csv: 1,558 rows
     program_id subapplicantStateAbbreviation  rows  coastal  with_uei
           bric                            CA   154       89       154
           bric                            OR    96       38        95
           bric                            WA   181      158       179
    lpdm_legacy                            CA    40       30         0
    lpdm_legacy                            WA    14        9         0
pdm_competitive                            CA   654      360         0
pdm_competitive                            OR   109       33         0
pdm_competitive                            WA   243      160         0
        pdm_cpf                            CA    43       33        16
        pdm_cpf                            OR     4        2         0
        pdm_cpf                            WA    20       14         6


In [22]:
ext["coastal_any"] = ext["any_coastal_county"] | ext.get("zip_in_coastal_county", False)
summ = ext.groupby(["program_id", "subapplicantStateAbbreviation"]).agg(
    rows=("program_id", "size"), in_window=("in_study_window", "sum"),
    coastal_county=("any_coastal_county", "sum"), coastal_county_or_zip=("coastal_any", "sum"),
    with_uei=("has_uei", "sum"), with_title=("has_title", "sum"))
summ = summ.astype(int)
pd.concat([summ, summ.sum().to_frame(("total", "")).T])

rows  in_window  coastal_county  coastal_county_or_zip  with_uei  with_title
bric            CA   154        154              89                     89       154         137
                OR    96         96              38                     38        95          67
                WA   181        181             158                    159       179         155
lpdm_legacy     CA    40         11              30                     31         0           0
                WA    14          1               9                     12         0           0
pdm_competitive CA   654        285             360                    414         0           0
                OR   109         53              33                     40         0           0
                WA   243        159             160                    198         0           0
pdm_cpf         CA    43         43              33                     33        16           0
                OR     4          4               2                      2         0           0
                WA    20         20              14                     17         6           0
total               1558       1007             926                   1033       450         359